# DATA VISUALISATION - PROJECT I: NHGIS and the US county population

This notebook builds maps and charts of the **2010 US county population by racial group**, using IPUMS NHGIS data (2020 county shapefile + time-series table CW8).

- **Task 1:** population barplots for a county (Los Angeles)
- **Task 2:** static choropleth map of all counties
- **Task 3:** interactive Leaflet-style map (Folium)
- **Task 4:** interactive dashboard (Dash, the Python equivalent of Shiny)

**Requirements:** see `requirements.txt` and the project `README.md`. An IPUMS API key is required (stored in `ipums_key.txt`, which is not tracked by git). Expected layout: this notebook lives in `notebooks/` and the downloaded data goes to `data/`.

## PRELIMINARIES

### Importing libraries

In [ ]:
## Data manipulation
import pandas as pd              # tables (DataFrame)
import numpy as np               # numerical computing

## Spatial data
import geopandas as gpd          # shapefiles + geographic data

## Data visualisation
import plotly.express as px      # interactive charts (bar charts, maps)
import folium                    # Leaflet-style interactive maps
import branca.colormap as cm     # colour palettes for maps
import matplotlib.pyplot as plt

## Interactive dashboard (Python equivalent of Shiny)
import dash
from dash import Dash, html, dcc, Input, Output

## NHGIS (IPUMS) API
from ipumspy import IpumsApiClient, extract_from_dict

import zipfile # Handling ZIP archives (reading, extracting, creating)
import os      # Interact with the operating system (files, folders, paths)
import glob    # Search for files using patterns (e.g. *.csv)
from matplotlib.ticker import FuncFormatter # Customise axis tick formatting in Matplotlib charts
from ipumspy import IpumsApiClient          # Connect to the IPUMS API to retrieve data
from ipumspy.api.extract import TimeSeriesTable # Define and manage time-series data extracts through IPUMS
import matplotlib.patches as mpatches # Graphical objects (patches) to enrich Matplotlib visualisations
import io     # Handle in-memory data streams like files (without writing to disk)
import base64 # Encode and decode data in Base64 format

### Connecting to the IPUMS NHGIS API

In [ ]:
# Read the API key stored locally in a text file
# (ipums_key.txt is listed in .gitignore: never commit your key)

with open("ipums_key.txt", "r") as f:
    IPUMS_API_KEY = f.read().strip()

# Connect to the API to submit requests

ipums = IpumsApiClient(IPUMS_API_KEY)

### Extraction

In [ ]:
# Request and download the 2020 county shapefile extract.
extract_shape = extract_from_dict({
    "collection": "nhgis",
    "description": "US counties shapefile",
    "shapefiles": ["us_county_2020_tl2020"],
})
extract_shape = ipums.submit_extract(extract_shape)
ipums.wait_for_extract(extract_shape)
os.makedirs("../data/nhgis_unzipped", exist_ok=True)
ipums.download_extract(extract_shape, download_dir="../data/")
shape_zips = sorted(glob.glob("../data/nhgis*_shape.zip"), key=os.path.getmtime)
if not shape_zips:
    raise FileNotFoundError("No NHGIS shapefile ZIP found in ../data after download")
with zipfile.ZipFile(shape_zips[-1]) as archive:
    archive.extractall("../data/nhgis_unzipped/")


In [ ]:
# Extract the county shapefile archive supplied inside the NHGIS download.
inner_zips = glob.glob("../data/nhgis_unzipped/**/*shapefile_tl2020_us_county_2020.zip", recursive=True)
if not inner_zips:
    raise FileNotFoundError("Could not find the 2020 county shapefile archive in ../data/nhgis_unzipped")
os.makedirs("../data/shapefile_final", exist_ok=True)
with zipfile.ZipFile(sorted(inner_zips)[-1]) as archive:
    archive.extractall("../data/shapefile_final/")


In [ ]:
# Import the US counties shapefile

shp_path = '../data/shapefile_final/US_county_2020.shp'  # Path to the US counties shapefile

shape_us = gpd.read_file(shp_path)  # Read the shapefile with GeoPandas into a GeoDataFrame

shape_us.head()  # Display the first rows of the GeoDataFrame to check its content

In [ ]:
# Keep only the contiguous USA (Alaska and Hawaii excluded)

shape_us["STATEFP"] = shape_us["STATEFP"].astype(int)

shape_us = shape_us[
    (shape_us["STATEFP"] < 57) &
    (~shape_us["STATEFP"].isin([2, 15]))
].copy()

shape_us.head()

In [ ]:
# Define and submit the county-level CW8 extract.
cw8 = TimeSeriesTable(name="CW8", years=[2000, 2010], geog_levels=["county"])
extract = extract_from_dict({
    "collection": "nhgis",
    "description": "CW8 county population extract",
    "time_series_tables": [cw8],
})
extract = ipums.submit_extract(extract)
ipums.wait_for_extract(extract)
ipums.download_extract(extract, download_dir="../data/")
os.makedirs("../data/cw8", exist_ok=True)
csv_zips = sorted(glob.glob("../data/nhgis*_csv.zip"), key=os.path.getmtime)
if not csv_zips:
    raise FileNotFoundError("No NHGIS CW8 CSV ZIP found in ../data after download")
with zipfile.ZipFile(csv_zips[-1]) as archive:
    archive.extractall("../data/cw8/")


In [ ]:
# Read the county-level CSV from the downloaded CW8 archive.
csv_files = glob.glob("../data/cw8/**/*ts_geog2010_county.csv", recursive=True)
if not csv_files:
    raise FileNotFoundError("Could not find the CW8 county CSV under ../data/cw8")
cw8_df = pd.read_csv(sorted(csv_files)[-1], encoding="latin1")
cw8_df.head()


### Merging the datasets

In [ ]:
cw8_df = cw8_df[cw8_df["GISJOIN"].isin(shape_us["GISJOIN"])]  # Keep only the cw8_df rows whose GISJOIN identifiers exist in shape_us

shape_us = shape_us[shape_us["GISJOIN"].isin(cw8_df["GISJOIN"])]  # Keep only the shape_us areas that are present in cw8_df

In [ ]:
cw8_2010 = cw8_df[[col for col in cw8_df.columns if "2010" in col or col == "GISJOIN"]].copy()  # Keep the columns containing "2010" and "GISJOIN", then make an independent copy

In [ ]:
cw8_2010["total_pop"] = cw8_2010.filter(like="CW8").sum(axis=1)  # Sum the columns containing "CW8" for each row to get the total population

In [ ]:
# CW8 has male (AA-AF) and female (AG-AL) counts. Combine sexes;
# the final two source categories are grouped as "Other".
column_groups = {
    "pop_white": ["CW8AA2010", "CW8AG2010"],
    "pop_black": ["CW8AB2010", "CW8AH2010"],
    "pop_native": ["CW8AC2010", "CW8AI2010"],
    "pop_asian": ["CW8AD2010", "CW8AJ2010"],
    "pop_other": ["CW8AE2010", "CW8AF2010", "CW8AK2010", "CW8AL2010"],
}
missing_columns = sorted({c for cols in column_groups.values() for c in cols} - set(cw8_2010.columns))
if missing_columns:
    raise KeyError(f"Expected CW8 2010 columns are missing: {missing_columns}")
for group, columns in column_groups.items():
    cw8_2010[group] = cw8_2010[columns].sum(axis=1)
cw8_columns = [c for c in cw8_2010.columns if c.startswith("CW8") and c.endswith("2010")]
cw8_2010["total_pop"] = cw8_2010[cw8_columns].sum(axis=1)


In [ ]:
# Create a new DataFrame with only the useful columns (identifier, population by group + total)

cw8_clean = cw8_2010[[
    "GISJOIN",
    "pop_white",
    "pop_black",
    "pop_native",
    "pop_asian",
    "pop_other",
    "total_pop"
]]

In [ ]:
shape_us = shape_us.merge(cw8_clean, on="GISJOIN", how="inner")  # Merge shape_us with cw8_clean on GISJOIN, keeping only the matching rows

In [ ]:
# Select and reorder the useful columns, including geographic information and population data

shape_us = shape_us[[

    "GISJOIN",
    "STATEFP",
    "COUNTYFP",
    "COUNTYNS",
    "GEOID",
    "NAME",
    "NAMELSAD",
    "LSAD",
    "CLASSFP",
    "MTFCC",
    "geometry",
    "pop_white",
    "pop_black",
    "pop_native",
    "pop_asian",
    "pop_other",
    "total_pop"
]]

### Task 1 : County population barplots

In [ ]:
# Create the chart

la = shape_us.loc[shape_us["GEOID"] == "06037"].copy()  # Select Los Angeles County using its GEOID and make a copy
row = la.iloc[0]  # Get the first (and only) row of the filtered DataFrame

labels = ["White", "Black", "Native", "Asian", "Other"]  # Racial categories to display

counts = [
    row["pop_white"],  # White population
    row["pop_black"],  # Black population
    row["pop_native"],  # Native American population
    row["pop_asian"],  # Asian population
    row["pop_other"]  # Population of other groups
]

colors = ["#4C78A8", "#F58518", "#54A24B", "#E45756", "#B279A2"]  # Colours associated with each group

fig, axes = plt.subplots(1, 2, figsize=(16,6))  # Create a figure with two side-by-side charts

def sci_format(x, pos):  # Function to format numbers in scientific notation
    return f"{x:.0e}"  # Convert values to scientific format (e.g. 1e6)

def add_labels(ax, bars):  # Function to add labels on the bars
    max_val = max(counts)  # Get the maximum value to adjust label positioning
    for bar, label in zip(bars, labels):  # Loop over each bar and its label
        width = bar.get_width()  # Get the bar length

        if width < max_val * 0.12:  # If the bar is small
            ax.text(width, bar.get_y() + bar.get_height()/2,
                    label, va='center', ha='left', fontweight='bold')  # Place the text outside the bar
        else:
            ax.text(width/2, bar.get_y() + bar.get_height()/2,
                    label, va='center', ha='center',
                    color='white', fontweight='bold')  # Place the text in the centre of the bar

bars1 = axes[0].barh(labels, counts, color=colors)  # Create a horizontal bar chart of the populations

axes[0].set_title("County : Los Angeles")  # Title of the first chart
axes[0].set_xlabel("Population")  # x-axis label
axes[0].set_ylabel("Race")  # y-axis label
axes[0].set_yticks([])  # Remove y-axis ticks for a cleaner look
axes[0].xaxis.set_major_formatter(FuncFormatter(sci_format))  # Apply scientific format to the x-axis

add_labels(axes[0], bars1)  # Add labels on the bars of the first chart

total = row["total_pop"]  # Total population
shares = [c / total for c in counts]  # Compute each group's share of the total population

bars2 = axes[1].barh(labels, shares, color=colors)  # Create a horizontal bar chart of the population shares

axes[1].set_xlabel("Population share")  # x-axis label of the second chart

axes[1].set_yticks([])  # Remove y-axis ticks for the second chart

plt.tight_layout()  # Automatically adjust spacing to avoid overlaps
plt.show()  # Display the figure

### Task 2 : All counties

In [ ]:
# Create the map

gdf = shape_us.copy()  # Copy the original GeoDataFrame to avoid modifying it directly

gdf["total_pop_class"] = gdf["total_pop"]  # Create a new column based on the total population

gdf["total_pop_class"] = gdf["total_pop_class"].fillna(-1)  # Replace missing values with -1 to identify them

bins = [
    0,  # Start of the first class
    7036,  # Threshold of the first class
    13009,  # Threshold of the second class
    20680,  # Threshold of the third class
    32542,  # Threshold of the fourth class
    54701,  # Threshold of the fifth class
    131842,  # Threshold of the sixth class
    gdf["total_pop"].max()  # Maximum value for the last class
]

fig, ax = plt.subplots(1, 1, figsize=(10,6))  # Create a figure and an axis for the map

gdf[gdf["total_pop_class"] != -1].plot(
    column="total_pop",  # Column used for colouring
    cmap="YlOrBr",  # Colour palette (yellow → brown)
    scheme="UserDefined",  # Use manually defined classes
    classification_kwds={"bins": bins[1:-1]},  # Class boundaries (without min and max)
    legend=True,  # Show the legend
    edgecolor="grey",  # County border colour
    linewidth=0.2,  # Border width
    ax=ax,  # Axis to draw on
    legend_kwds={
        "loc": "lower left",  # Legend position
        "fontsize": 8,  # Text size
        "frameon": True,  # Draw a frame around the legend
        "title": ""  # No legend title
    }
)

missing_data = gdf[gdf["total_pop_class"] == -1]  # Select observations with missing data

if not missing_data.empty:
    missing_data.plot(
        color="lightgrey",  # Colour areas without data in light grey
        ax=ax  # Draw on the same axis
    )

leg = ax.get_legend()  # Get the existing legend

if not missing_data.empty:
    missing_patch = mpatches.Patch(color="lightgrey", label="Missing")  # Create a legend item for missing data

    handles = leg.legendHandles  # Get the legend's graphical elements
    labels = [t.get_text() for t in leg.texts]  # Get the associated labels

    handles.append(missing_patch)  # Add the "Missing" patch to the elements
    labels.append("Missing")  # Add the corresponding label

    ax.legend(
        handles,
        labels,
        loc="lower left",  # Legend position
        fontsize=8,  # Text size
        frameon=True  # Enable the legend frame
    )

ax.set_title("Total Population by County", fontsize=12)  # Map title
ax.axis("off")  # Remove axes for a clean map look

plt.tight_layout()  # Automatically adjust margins
plt.show()  # Display the map

### Task 3 : leaflet

In [ ]:
# Create the interactive map

gdf = shape_us.to_crs("EPSG:4326").copy()  # Convert the coordinate system to WGS84 (web format) and make a copy

gdf["geometry"] = gdf["geometry"].simplify(0.01)  # Simplify geometries to lighten the map and improve performance

m = folium.Map(
    location=[37.8, -96],  # Centre the map on the United States
    zoom_start=4,  # Initial zoom level
    tiles="cartodbpositron"  # Use a light, minimalist basemap
)

colormap = cm.linear.YlOrBr_09.scale(
    gdf["total_pop"].min(),  # Minimum value of the colour scale
    gdf["total_pop"].max()  # Maximum value of the colour scale
)

colormap.caption = "Population by county"  # Add a title to the legend

def style_function(feature):  # Define the style of the GeoJSON polygons
    pop = feature["properties"]["total_pop"]  # Get the county population
    return {
        "fillColor": colormap(pop),  # Apply a colour according to the population
        "color": "black",  # Border colour
        "weight": 0.25,  # Border width
        "fillOpacity": 0.75  # Fill transparency
    }

geojson = folium.GeoJson(
    gdf,  # Convert the GeoDataFrame into a GeoJSON layer
    style_function=style_function,  # Apply the style function

    tooltip=folium.GeoJsonTooltip(
        fields=["NAME", "total_pop"],  # Fields shown on hover
        aliases=["County:", "Population:"],  # Displayed labels
        sticky=True  # Keep the tooltip visible while hovering
    ),

    popup=folium.GeoJsonPopup(
        fields=["NAME", "total_pop"],  # Fields shown on click
        aliases=["County:", "Population:"]  # Displayed labels
    )
)

geojson.add_to(m)  # Add the GeoJSON layer to the map

colormap.add_to(m)  # Add the legend (colour scale) to the map

m.save("leaflet_task3_20_20_fast.html")  # Export the interactive map as an HTML file

print("✔ Leaflet ready ")  # Print a confirmation message
m  # Display the map in the Python environment

### Task 4 : Shiny

The dashboard is launched by the cell below (local server). Preview:

![Dashboard - California / Los Angeles](../img/dashboard_california.png)

![Dashboard - another state](../img/dashboard_other_state.png)

In [ ]:
# Dash dashboard. County GEOIDs are used as values so duplicate county names
# in different states cannot select the wrong county.
state_names = {
    "01":"Alabama", "02":"Alaska", "04":"Arizona", "05":"Arkansas", "06":"California",
    "08":"Colorado", "09":"Connecticut", "10":"Delaware", "11":"District of Columbia",
    "12":"Florida", "13":"Georgia", "15":"Hawaii", "16":"Idaho", "17":"Illinois",
    "18":"Indiana", "19":"Iowa", "20":"Kansas", "21":"Kentucky", "22":"Louisiana",
    "23":"Maine", "24":"Maryland", "25":"Massachusetts", "26":"Michigan", "27":"Minnesota",
    "28":"Mississippi", "29":"Missouri", "30":"Montana", "31":"Nebraska", "32":"Nevada",
    "33":"New Hampshire", "34":"New Jersey", "35":"New Mexico", "36":"New York",
    "37":"North Carolina", "38":"North Dakota", "39":"Ohio", "40":"Oklahoma", "41":"Oregon",
    "42":"Pennsylvania", "44":"Rhode Island", "45":"South Carolina", "46":"South Dakota",
    "47":"Tennessee", "48":"Texas", "49":"Utah", "50":"Vermont", "51":"Virginia",
    "53":"Washington", "54":"West Virginia", "55":"Wisconsin", "56":"Wyoming"
}
gdf = shape_us.copy()
gdf["STATEFP"] = gdf["STATEFP"].astype(str).str.zfill(2)
gdf["STATE_NAME"] = gdf["STATEFP"].map(state_names)
gdf = gdf.dropna(subset=["STATE_NAME"])
states = sorted(gdf["STATE_NAME"].unique())
app = dash.Dash(__name__)
app.layout = html.Div([
    html.H1("US County Population 2010"),
    html.Div([
        html.Div([html.Label("State"), dcc.Dropdown(id="state", options=[{"label":s,"value":s} for s in states], value="California"),
                  html.Br(), html.Label("County"), dcc.Dropdown(id="county")],
                 style={"width":"20%","display":"inline-block","verticalAlign":"top"}),
        html.Div([html.Img(id="map"), html.Br(), html.Img(id="barplot")], style={"width":"75%","display":"inline-block"})
    ])
])
@app.callback(Output("county","options"), Output("county","value"), Input("state","value"))
def update_county(state):
    dff=gdf[gdf["STATE_NAME"]==state].sort_values("NAME")
    options=[{"label":f"{name} County","value":geoid} for name,geoid in zip(dff["NAME"],dff["GEOID"])]
    if not options: return [], None
    default="06037" if "06037" in dff["GEOID"].values else options[0]["value"]
    return options, default
def create_map(state):
    dff=gdf[gdf["STATE_NAME"]==state].copy(); dff["total_pop_class"]=dff["total_pop"].fillna(-1)
    fig,ax=plt.subplots(figsize=(6,5)); available=dff[dff["total_pop_class"]!=-1]
    if not available.empty:
        available.plot(column="total_pop",cmap="YlOrBr",scheme="UserDefined",
                       classification_kwds={"bins":[7036,13009,20680,32542,54701,131842]},ax=ax)
    missing=dff[dff["total_pop_class"]==-1]
    if not missing.empty: missing.plot(color="lightgrey",ax=ax)
    ax.set_title(f"{state} - Population by County"); ax.axis("off")
    buf=io.BytesIO(); fig.savefig(buf,format="png",bbox_inches="tight"); plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()
def create_barplot(geoid):
    matches=gdf[gdf["GEOID"]==geoid]
    if matches.empty: return ""
    row=matches.iloc[0]; labels=["White","Black","Native","Asian","Other"]
    counts=[row["pop_white"],row["pop_black"],row["pop_native"],row["pop_asian"],row["pop_other"]]
    colors=["#4C78A8","#F58518","#54A24B","#E45756","#B279A2"]
    fig,axes=plt.subplots(1,2,figsize=(10,4)); bars=axes[0].barh(labels,counts,color=colors)
    axes[0].set_title(f"{row['NAME']} County"); axes[0].set_yticks([])
    axes[0].xaxis.set_major_formatter(FuncFormatter(lambda x,pos:f"{x:.0e}")); max_val=max(counts)
    for bar,label in zip(bars,labels):
        width=bar.get_width(); inside=width>=max_val*0.12
        axes[0].text(width/2 if inside else width,bar.get_y()+bar.get_height()/2,label,va="center",
                     ha="center" if inside else "left",color="white" if inside else "black",fontweight="bold")
    axes[1].barh(labels,[c/row["total_pop"] for c in counts],color=colors); axes[1].set_yticks([])
    axes[1].set_xlabel("Population share"); fig.tight_layout()
    buf=io.BytesIO(); fig.savefig(buf,format="png",bbox_inches="tight"); plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()
@app.callback(Output("map","src"),Output("barplot","src"),Input("state","value"),Input("county","value"))
def update_dashboard(state,county):
    if not state or not county: return "", ""
    return f"data:image/png;base64,{create_map(state)}", f"data:image/png;base64,{create_barplot(county)}"
# In a notebook, run the local dashboard with app.run(debug=False).
app.run(debug=False)
